CELL 1: IMPORT THƯ VIỆN

In [ ]:
# Dữ liệu 
import pandas as pd
import numpy as np

# Ảnh & PyTorch 
import torch
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Tìm kiếm tương đồng
import faiss
from sklearn.metrics.pairwise import cosine_similarity

# Vẽ biểu đồ 
import matplotlib.pyplot as plt
import seaborn as sns

# Tiện ích
import os
from tqdm import tqdm

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 20)      
plt.rcParams['figure.figsize'] = (10, 5)    
sns.set_style('whitegrid')                  


print('Import thư viện thành công!')
print(f'PyTorch version : {torch.__version__}')
# Kiểm tra máy có GPU không. GPU giúp chạy nhanh hơn CPU rất nhiều.
# Laptop bình thường thường sẽ hiện 'cpu' – không sao, vẫn chạy được.
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

CELL 2: ĐƯỜNG DẪN DỮ LIỆU

In [ ]:
DATA_DIR  = r'D:\New folder (3)\project\data\raw'  

CSV_PATH  = os.path.join(DATA_DIR, 'train.csv')
IMAGE_DIR = os.path.join(DATA_DIR, 'train_images')

# Thư mục lưu kết quả – tạo tự động nếu chưa có
PROCESSED = '../data/processed/'
RESULTS   = '../results/'
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(RESULTS,   exist_ok=True)

# Kiểm tra file có tồn tại không
for p in [CSV_PATH, IMAGE_DIR]:
    status = 'Đã tìm thấy thư mục' if os.path.exists(p) else 'Không tìm thấy – kiểm tra lại đường dẫn!'
    print(f'{status}  {p}')

CELL 3: LOAD CANDIDATE_DF CHUNG

In [ ]:
SEED = 42
CANDIDATE_PATH = os.path.join(PROCESSED, 'candidate_df_tuan3.csv')

# Đọc train.csv (cần khi tạo mới candidate_df)
df = pd.read_csv(CSV_PATH)
print(f'Đã đọc train.csv: {df.shape[0]:,} dòng, {df.shape[1]} cột')

if os.path.exists(CANDIDATE_PATH):
    candidate_df = pd.read_csv(CANDIDATE_PATH)
    print(f'Đã load candidate_df từ file: {CANDIDATE_PATH}')
else:
    print('Chưa có file candidate_df — đang tạo mới (250 nhóm × 2 ảnh = 500 ảnh)...')
    so_anh_moi_nhom = df['label_group'].value_counts()
    nhom_hop_le = so_anh_moi_nhom[so_anh_moi_nhom >= 2].index
    df_loc = df[df['label_group'].isin(nhom_hop_le)].copy()

    EVAL_GROUPS = 250
    valid_groups = df_loc['label_group'].drop_duplicates().sample(
        n=min(EVAL_GROUPS, df_loc['label_group'].nunique()),
        random_state=SEED,
    )

    parts = []
    for nhom in valid_groups:
        parts.append(df_loc[df_loc['label_group'] == nhom].sample(n=2, random_state=SEED))

    candidate_df = pd.concat(parts, ignore_index=True)
    candidate_df = candidate_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    candidate_df.to_csv(CANDIDATE_PATH, index=False)
    print(f'Đã lưu candidate_df: {CANDIDATE_PATH}')

nhom_counts = candidate_df['label_group'].value_counts()
print(f'\nThống kê candidate_df (dùng chung cho mọi baseline):')
print(f'  Tổng ảnh        : {len(candidate_df):,}')
print(f'  Số nhóm         : {candidate_df["label_group"].nunique():,}')
print(f'  Nhóm đúng 2 ảnh : {(nhom_counts == 2).sum():,}')
print(f'  Nhóm chỉ 1 ảnh  : {(nhom_counts == 1).sum():,}  ← lý tưởng = 0')
print(f'  Cột dữ liệu     : {list(candidate_df.columns)}')
candidate_df.head(3)

CELL 4: RESNET50 + FAISS( BẢO )

CELL 5: CLIP ( HƯNG )

CELL 6: KẾT HỢP IMAGE + TEXT ( VỸ )

In [ ]:
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

TOP_K_PRED = 5
K_LIST = [1, 3, 5, 10]
MAX_K = max(K_LIST)
ALPHA_WEIGHTED = 0.6  # TH4: 0.6 * Image + 0.4 * Text (chưa fine-tune)

labels = candidate_df['label_group'].values
n = len(candidate_df)
_img_col = 'query_image' if 'query_image' in candidate_df.columns else 'image'
image_names = candidate_df[_img_col].tolist()


def average_precision(ranked_labels, true_label, total_relevant):
    if total_relevant == 0:
        return 0.0
    ap, hits = 0.0, 0
    for rank, label in enumerate(ranked_labels, start=1):
        if label == true_label:
            hits += 1
            ap += hits / rank
    return ap / total_relevant


def build_metrics_df(sim_matrix: np.ndarray, method_name: str) -> pd.DataFrame:
    rows, all_ap = [], []
    for i in range(n):
        total_relevant = sum(1 for j in range(n) if labels[j] == labels[i] and j != i)
        if total_relevant == 0:
            continue
        top_idx = np.argsort(-sim_matrix[i])[:MAX_K]
        ranked_labels = [labels[j] for j in top_idx]
        ap = average_precision(ranked_labels, labels[i], total_relevant)
        all_ap.append(ap)
        row = {'method': method_name, 'K': None}
        for k in K_LIST:
            hits = sum(1 for lbl in ranked_labels[:k] if lbl == labels[i])
            row[f'p@{k}'] = round(hits / k, 4)
            row[f'r@{k}'] = round(hits / min(total_relevant, k), 4)
        rows.append(row)

    mAP = round(float(np.mean(all_ap)), 4)
    summary = []
    for k in K_LIST:
        p_col = [r[f'p@{k}'] for r in rows]
        r_col = [r[f'r@{k}'] for r in rows]
        summary.append({
            'method': method_name,
            'K': k,
            'Precision@K': round(float(np.mean(p_col)), 4),
            'Recall@K': round(float(np.mean(r_col)), 4),
            'mAP': mAP,
        })
    return pd.DataFrame(summary)


def top_k_predictions(sim_matrix: np.ndarray, k: int = TOP_K_PRED) -> list:
    preds = []
    for i in range(n):
        top_idx = np.argsort(-sim_matrix[i])[:k]
        preds.append([image_names[j] for j in top_idx])
    return preds


# ── TH1: Text only (TF-IDF — giống Baseline3) ───────────────────────────────
titles = candidate_df['title'].fillna('').astype(str)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=1)
tfidf_matrix = vectorizer.fit_transform(titles)
sim_text = cosine_similarity(tfidf_matrix).astype(np.float32)
np.fill_diagonal(sim_text, -1.0)

pred_text = top_k_predictions(sim_text, TOP_K_PRED)
metrics_th1 = build_metrics_df(sim_text, 'TH1_text_only')
metrics_th1.to_csv(os.path.join(RESULTS, 'tuan3_tfidf_metrics.csv'), index=False)
print(f'TH1 (Text): mAP = {metrics_th1["mAP"].iloc[0]:.4f}')

# ── TH2: Image only (ResNet50 — giống Tuan2 / dùng feature_matrix nếu CELL 4 đã chạy) ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if 'feature_matrix' not in globals():
    print('Chưa có feature_matrix — trích xuất ResNet50 trong cell này...')
    transform_pipeline = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    def _load_tensor(path):
        img = Image.open(path).convert('RGB')
        return transform_pipeline(img).unsqueeze(0)

    if 'feature_extractor' not in globals():
        _model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        _model.eval().to(device)
        feature_extractor = torch.nn.Sequential(*list(_model.children())[:-1])

    feats = []
    with torch.no_grad():
        for fname in image_names:
            t = _load_tensor(os.path.join(IMAGE_DIR, fname)).to(device)
            feats.append(feature_extractor(t).view(1, -1).cpu())
    feature_matrix = torch.cat(feats, dim=0)

features_norm = F.normalize(feature_matrix.float(), p=2, dim=1)
sim_image = (features_norm @ features_norm.T).numpy().astype(np.float32)
np.fill_diagonal(sim_image, -1.0)

pred_image = top_k_predictions(sim_image, TOP_K_PRED)
metrics_th2 = build_metrics_df(sim_image, 'TH2_image_only')
metrics_th2.to_csv(os.path.join(RESULTS, 'tuan3_image_metrics.csv'), index=False)
print(f'TH2 (Image): mAP = {metrics_th2["mAP"].iloc[0]:.4f}')

# ── TH3: Late fusion — Concat vector L2-normalized ────────────────────────────
img_np = features_norm.numpy()
txt_np = sk_normalize(tfidf_matrix.toarray(), norm='l2')
combined = sk_normalize(np.hstack([img_np, txt_np]), norm='l2')
sim_concat = (combined @ combined.T).astype(np.float32)
np.fill_diagonal(sim_concat, -1.0)

pred_fusion = top_k_predictions(sim_concat, TOP_K_PRED)
metrics_th3 = build_metrics_df(sim_concat, 'TH3_concat_image_text')
metrics_th3.to_csv(os.path.join(RESULTS, 'tuan3_imagetext_metrics.csv'), index=False)
print(f'TH3 (Concat image+text): mAP = {metrics_th3["mAP"].iloc[0]:.4f}')

# ── TH4: Weighted ensemble (thay fine-tune) ─────────────────────────────────
sim_weighted = (ALPHA_WEIGHTED * sim_image + (1 - ALPHA_WEIGHTED) * sim_text).astype(np.float32)
np.fill_diagonal(sim_weighted, -1.0)

pred_weighted = top_k_predictions(sim_weighted, TOP_K_PRED)
metrics_th4 = build_metrics_df(sim_weighted, f'TH4_weighted_{ALPHA_WEIGHTED:.1f}img')
metrics_th4.to_csv(os.path.join(RESULTS, 'tuan3_weighted_metrics.csv'), index=False)
print(f'TH4 (Weighted {ALPHA_WEIGHTED:.1f}·Image + {1-ALPHA_WEIGHTED:.1f}·Text): mAP = {metrics_th4["mAP"].iloc[0]:.4f}')

# ── Bảng candidate_df: 3 cột prediction Top-5 (theo CLAUDE.md) ───────────────
# Giữ tên file gốc ở query_image; 3 cột image/text/image+text = kết quả truy xuất
candidate_df = candidate_df.copy()
if 'query_image' not in candidate_df.columns:
    candidate_df = candidate_df.rename(columns={'image': 'query_image'})

candidate_df['image'] = [' | '.join(p) for p in pred_image]
candidate_df['text'] = [' | '.join(p) for p in pred_text]
candidate_df['image+text'] = [' | '.join(p) for p in pred_fusion]

candidate_df.to_csv(os.path.join(PROCESSED, 'candidate_df_predictions.csv'), index=False)
print(f'\nĐã lưu: {os.path.join(PROCESSED, "candidate_df_predictions.csv")}')
print('  query_image = ảnh query | image/text/image+text = Top-5 retrieval')

# Tóm tắt 4 phương án
compare_4 = pd.concat([metrics_th1, metrics_th2, metrics_th3, metrics_th4], ignore_index=True)
compare_4_pivot = compare_4.pivot_table(index='method', values='mAP', aggfunc='first').sort_values('mAP', ascending=False)
print('\n=== mAP 4 PHƯƠNG ÁN (CELL 6) ===')
print(compare_4_pivot.to_string())
candidate_df[['query_image', 'title', 'label_group', 'image', 'text', 'image+text']].head(3)

CELL 7 BẢNG SO SÁNH 4 PHƯƠNG PHÁP

CELL 8: PHÂN TÍCH LỖI

CELL 9: BIỂU ĐỒ SO SÁNH